> **Status: PREPARED — NOT YET EXECUTED**  
> Skeleton notebook ready. Requires Assignment 1 outputs in `01_input_graph/`.  
> No results fabricated. All metric cells have working code, pending input data.

# MaCAD S.3 — Assignment 2
## A2-01: Graph Metric Computation — Double-L Residential Building

**Assignment objective:** Compute a full suite of graph-theoretic spatial metrics on the
room-and-circulation graph produced in Assignment 1. Reproduce the S03 spatial
intelligence workflow using the resident_gen graph rather than the course's grid shell.

**Course workflow:** S03 — Spatial Intelligence / Graph Analysis  
**Primary reference notebooks (read-only, in `class_notebooks/S03_graph_analysis/`):**

| Notebook | Metrics covered |
|----------|----------------|
| `S03-07 Spatial Intelligence Part 1.ipynb` | Closeness centrality, betweenness centrality, density, diameter, shortest paths, `Graph.CompiledRoutingGraph` |
| `S03-08 Spatial Intelligence Part 2.ipynb` | Community detection, degree centrality, `Graph.CommunityPartition`, `Graph.BinByDictionaryKey` |
| `S03-09 Spatial Intelligence Part 3.ipynb` | Isovist / visibility analysis (geometry-dependent — not applicable here) |

---

**Inputs (from Assignment 1 `04_graph_dataset/`):**

| File | Rows | Description |
|------|------|-------------|
| `nodes.csv` | ~596 | Node attributes: node_id, node_role, space_type, floor_id, area, X, Y, Z |
| `edges.csv` | TBD | Edge list: src_id, dst_id, edge_type |

**Outputs:**

| File | Location | Description |
|------|----------|-------------|
| `metrics_table.csv` | `03_results/` | Per-node: degree, betweenness, closeness, clustering, community |
| `graph_summary.csv` | `03_results/` | Graph-level: density, diameter, avg path length |
| Metric visualisations | `04_visuals/` | One PNG per metric, coloured plan view |

### Assignment 2 Requirement Checklist

| # | Metric | Course API | Status |
|---|--------|-----------|--------|
| 1 | Load graph from Assignment 1 | — | \[ \] Run to confirm |
| 2 | Rebuild topologicpy Graph from XYZ + edge list | `Graph.ByVerticesEdges` | \[ \] Run to confirm |
| 3 | Graph density | `Graph.Density(g)` | \[ \] Run to confirm |
| 4 | Graph diameter | `Graph.Diameter(g)` | \[ \] Run to confirm |
| 5 | Degree centrality | `Graph.DegreeCentrality(g)` | \[ \] Run to confirm |
| 6 | Betweenness centrality | `Graph.BetweennessCentrality(g, normalize=True)` | \[ \] Run to confirm |
| 7 | Closeness centrality | `Graph.ClosenessCentrality(g)` | \[ \] Run to confirm |
| 8 | Clustering coefficient | NetworkX `nx.clustering(G)` | \[ \] Run to confirm |
| 9 | Community detection | `Graph.CommunityPartition(g)` | \[ \] Run to confirm |
| 10 | Shortest path (3 selected pairs) | `Graph.ShortestPath(crg, vA, vB)` | \[ \] Run to confirm |
| 11 | Export metrics_table.csv + graph_summary.csv | — | \[ \] Run to confirm |
| 12 | One coloured plan PNG per metric | `Topology.Show` | \[ \] Run to confirm |

---
## 1. Setup

In [ ]:
# !pip install topologicpy

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge as TPEdge
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper

import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import warnings
warnings.filterwarnings("ignore")

print("topologicpy version:", Helper.Version())

In [ ]:
# ── Renderer ───────────────────────────────────────────────────────────────
renderer = "notebook"  # "notebook" | "vscode" | "browser"

# ── Path configuration ─────────────────────────────────────────────────────
PROJECT_ROOT = "D:/GitHub/GML_Edu"

A1_ROOT      = os.path.join(PROJECT_ROOT, "assignments", "assignment_01_graph_generation")
A1_DATASET   = os.path.join(A1_ROOT, "04_graph_dataset")

A2_ROOT      = os.path.join(PROJECT_ROOT, "assignments", "assignment_02_graph_analysis")
INPUT_DIR    = os.path.join(A2_ROOT, "01_input_graph")
RESULTS_DIR  = os.path.join(A2_ROOT, "03_results")
VISUALS_DIR  = os.path.join(A2_ROOT, "04_visuals")

os.makedirs(INPUT_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(VISUALS_DIR, exist_ok=True)

# Prefer local copies in 01_input_graph/; fall back to A1 outputs
NODES_CSV = (
    os.path.join(INPUT_DIR,  "nodes.csv") if os.path.exists(os.path.join(INPUT_DIR, "nodes.csv"))
    else os.path.join(A1_DATASET, "nodes.csv")
)
EDGES_CSV = (
    os.path.join(INPUT_DIR,  "edges.csv") if os.path.exists(os.path.join(INPUT_DIR, "edges.csv"))
    else os.path.join(A1_DATASET, "edges.csv")
)

print("Path check:")
for label, path in [("nodes.csv", NODES_CSV), ("edges.csv", EDGES_CSV)]:
    status = "OK" if os.path.exists(path) else "MISSING — run Assignment 1 first"
    print(f"  {label:12s}: [{status}]  {path}")

if not os.path.exists(NODES_CSV) or not os.path.exists(EDGES_CSV):
    raise FileNotFoundError(
        "Assignment 1 graph dataset not found.\n"
        "Run A1_01 + A1_02, then either:\n"
        "  (a) copy nodes.csv + edges.csv to 01_input_graph/, or\n"
        "  (b) leave them in place — this notebook reads A1's 04_graph_dataset/ directly."
    )

---
## 2. Load Graph from Assignment 1

In [ ]:
nodes = pd.read_csv(NODES_CSV)
edges = pd.read_csv(EDGES_CSV)

print(f"nodes.csv : {len(nodes)} rows")
print(f"edges.csv : {len(edges)} rows")
print(f"Node columns: {list(nodes.columns)}")
print()
print(nodes[["node_role", "space_type"]].value_counts().head(10))

In [ ]:
# Build NetworkX graph (for clustering coefficient and connectivity checks)
G_nx = nx.Graph()
for _, row in nodes.iterrows():
    G_nx.add_node(int(row["node_id"]), **row.to_dict())
for _, row in edges.iterrows():
    G_nx.add_edge(int(row["src_id"]), int(row["dst_id"]))

print(f"NetworkX: {G_nx.number_of_nodes()} nodes, {G_nx.number_of_edges()} edges")
print(f"Connected: {nx.is_connected(G_nx)}  |  Components: {nx.number_connected_components(G_nx)}")

# Largest connected component for path-dependent metrics
if not nx.is_connected(G_nx):
    lcc_nodes = max(nx.connected_components(G_nx), key=len)
    G_lcc     = G_nx.subgraph(lcc_nodes).copy()
    print(f"Largest CC : {len(G_lcc)} nodes (used for path-dependent metrics)")
else:
    G_lcc = G_nx
    print("Graph is connected — using full graph for all metrics.")

---
## 3. Rebuild topologicpy Graph

**Course pattern — S03-07:**  
The S03 notebooks work with a topologicpy `Graph` object (not NetworkX) to access
`Graph.ClosenessCentrality`, `Graph.BetweennessCentrality`, `Graph.DegreeCentrality`,
`Graph.CommunityPartition`, etc.

Since we are working from CSV (not live geometry), we rebuild the topologicpy graph
from the X, Y, Z centroid positions stored in `nodes.csv` and the edge list in `edges.csv`.

API: `Graph.ByVerticesEdges(vertices, edges)` — creates a graph directly from
topologic Vertex and Edge objects.

In [ ]:
# Build topologicpy Vertex objects from centroid positions
tp_verts = {}
for _, row in nodes.iterrows():
    nid = int(row["node_id"])
    v   = Vertex.ByCoordinates(float(row["X"]), float(row["Y"]), float(row["Z"]))
    # Carry key attributes in vertex dictionary
    d   = Dictionary.ByKeysValues(
        ["node_id", "node_role", "space_type", "floor_id", "area"],
        [nid,
         str(row.get("node_role", "")),
         str(row.get("space_type", "")),
         int(row.get("floor_id") or 0),
         float(row.get("area", 0))]
    )
    v = Topology.SetDictionary(v, d)
    tp_verts[nid] = v

print(f"Topologicpy vertices created: {len(tp_verts)}")

In [ ]:
# Build topologicpy Edge objects
tp_edges = []
for _, row in edges.iterrows():
    src = tp_verts.get(int(row["src_id"]))
    dst = tp_verts.get(int(row["dst_id"]))
    if src is not None and dst is not None:
        e = TPEdge.ByVertices([src, dst])
        if e is not None:
            tp_edges.append(e)

print(f"Topologicpy edges created: {len(tp_edges)}")

# Build Graph
tp_graph = Graph.ByVerticesEdges(list(tp_verts.values()), tp_edges)
tp_verts_list = Graph.Vertices(tp_graph)
tp_edges_list = Graph.Edges(tp_graph)

print(f"topologicpy Graph: {len(tp_verts_list)} vertices, {len(tp_edges_list)} edges")

---
## 4. Graph-Level Statistics

**Course APIs — S03-07:**
- `Graph.Density(g)` → float in [0, 1]
- `Graph.Diameter(g)` → int (longest shortest path)
- Average path length → NetworkX `nx.average_shortest_path_length(G_lcc)`

In [ ]:
# topologicpy graph-level stats
tp_density  = Graph.Density(tp_graph)
tp_diameter = Graph.Diameter(tp_graph)

# NetworkX equivalents
nx_density  = nx.density(G_nx)
nx_diameter = nx.diameter(G_lcc)
nx_avg_path = nx.average_shortest_path_length(G_lcc)

print("=" * 44)
print("GRAPH-LEVEL STATISTICS")
print("=" * 44)
print(f"  Nodes              : {G_nx.number_of_nodes()}")
print(f"  Edges              : {G_nx.number_of_edges()}")
print(f"  Density (tp)       : {tp_density:.4f}")
print(f"  Density (nx)       : {nx_density:.4f}")
print(f"  Diameter (tp)      : {tp_diameter}")
print(f"  Diameter (nx, LCC) : {nx_diameter}")
print(f"  Avg path len (LCC) : {nx_avg_path:.2f}")
print(f"  Connected comp.    : {nx.number_connected_components(G_nx)}")
print("=" * 44)

graph_summary = pd.DataFrame([{
    "n_nodes"      : G_nx.number_of_nodes(),
    "n_edges"      : G_nx.number_of_edges(),
    "density"      : tp_density,
    "diameter"     : tp_diameter,
    "avg_path_len" : nx_avg_path,
    "n_components" : nx.number_connected_components(G_nx),
}])
graph_summary.to_csv(os.path.join(RESULTS_DIR, "graph_summary.csv"), index=False)
print(f"Saved: {RESULTS_DIR}/graph_summary.csv")

---
## 5. Degree Centrality

**Course API — S03-08:**  
`Graph.DegreeCentrality(g, normalize=False)` — returns a list of values, one per vertex
(in the same order as `Graph.Vertices(g)`). Sets a `dc` key on each vertex dictionary.

In [ ]:
# topologicpy degree centrality (S03-08)
dc_list = Graph.DegreeCentrality(tp_graph, normalize=True)
print(f"Degree centrality values: {len(dc_list)}  (min={min(dc_list):.4f}, max={max(dc_list):.4f})")

# Map back to node_id using vertex position
vert_to_nid = {}
for _, row in nodes.iterrows():
    key = (round(float(row["X"]), 2), round(float(row["Y"]), 2), round(float(row["Z"]), 2))
    vert_to_nid[key] = int(row["node_id"])

dc_by_nid = {}
for v, dc_val in zip(tp_verts_list, dc_list):
    key = (round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2))
    nid = vert_to_nid.get(key)
    if nid is not None:
        dc_by_nid[nid] = dc_val

# NetworkX degree centrality (cross-check)
nx_dc = nx.degree_centrality(G_nx)

print("\nTop 5 nodes by degree centrality (topologicpy):")
top_dc = sorted(dc_by_nid, key=dc_by_nid.get, reverse=True)[:5]
for nid in top_dc:
    role = G_nx.nodes[nid].get("node_role", "?")
    st   = G_nx.nodes[nid].get("space_type", "?")
    print(f"  node {nid:4d}  role={role:10s}  type={st:10s}  dc={dc_by_nid[nid]:.4f}")

---
## 6. Betweenness Centrality

**Course API — S03-07:**  
`Graph.BetweennessCentrality(g, normalize=True, colorScale="thermal")` — returns list,
sets `bc` and `bc_color` keys on vertex dictionaries.

In [ ]:
print("Computing betweenness centrality (may take a moment for large graphs)...")
bc_list = Graph.BetweennessCentrality(tp_graph, normalize=True, colorScale="thermal")
print(f"Betweenness values: {len(bc_list)}  (min={min(bc_list):.4f}, max={max(bc_list):.4f})")

bc_by_nid = {}
for v, bc_val in zip(tp_verts_list, bc_list):
    key = (round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2))
    nid = vert_to_nid.get(key)
    if nid is not None:
        bc_by_nid[nid] = bc_val

print("\nTop 5 nodes by betweenness centrality:")
top_bc = sorted(bc_by_nid, key=bc_by_nid.get, reverse=True)[:5]
for nid in top_bc:
    role = G_nx.nodes[nid].get("node_role", "?")
    st   = G_nx.nodes[nid].get("space_type", "?")
    fl   = G_nx.nodes[nid].get("floor_id", "?")
    print(f"  node {nid:4d}  floor={fl}  role={role:10s}  type={st:10s}  bc={bc_by_nid[nid]:.4f}")

---
## 7. Closeness Centrality

**Course API — S03-07:**  
`Graph.ClosenessCentrality(g, colorScale="thermal")` — returns list,
sets `cc` and `cc_color` keys on vertex dictionaries.

In [ ]:
print("Computing closeness centrality...")
cc_list = Graph.ClosenessCentrality(tp_graph, colorScale="thermal")
print(f"Closeness values: {len(cc_list)}  (min={min(cc_list):.4f}, max={max(cc_list):.4f})")

cc_by_nid = {}
for v, cc_val in zip(tp_verts_list, cc_list):
    key = (round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2))
    nid = vert_to_nid.get(key)
    if nid is not None:
        cc_by_nid[nid] = cc_val

# NetworkX cross-check
nx_cc = nx.closeness_centrality(G_nx)

print("\nTop 5 nodes by closeness centrality (topologicpy):")
for nid in sorted(cc_by_nid, key=cc_by_nid.get, reverse=True)[:5]:
    role = G_nx.nodes[nid].get("node_role", "?")
    st   = G_nx.nodes[nid].get("space_type", "?")
    print(f"  node {nid:4d}  role={role:10s}  type={st:10s}  cc={cc_by_nid[nid]:.4f}")

---
## 8. Clustering Coefficient

**NetworkX API** (no direct topologicpy equivalent in S03):  
`nx.clustering(G)` — local clustering coefficient per node; measures triangles
in the adjacency neighbourhood (how 'clique-like' each room's neighbourhood is).

In [ ]:
clustering = nx.clustering(G_nx)
global_clust = nx.average_clustering(G_nx)

clust_vals = list(clustering.values())
print(f"Clustering coefficient:")
print(f"  Global average : {global_clust:.4f}")
print(f"  Max            : {max(clust_vals):.4f}")
print(f"  Min            : {min(clust_vals):.4f}")
print(f"  Zero-clustering: {sum(1 for v in clust_vals if v == 0)} nodes")
print()
print("Top 5 nodes by clustering coefficient:")
for nid, val in sorted(clustering.items(), key=lambda x: -x[1])[:5]:
    role = G_nx.nodes[nid].get("node_role", "?")
    st   = G_nx.nodes[nid].get("space_type", "?")
    print(f"  node {nid:4d}  role={role:10s}  type={st:10s}  clust={val:.4f}")

---
## 9. Community Detection

**Course API — S03-08:**  
`Graph.CommunityPartition(g, colorScale="thermal")` — Louvain community detection.
Returns list of community IDs (one per vertex), sets `community` and `cp_color`
keys on vertex dictionaries.

In [ ]:
print("Running community detection (Graph.CommunityPartition — S03-08)...")
comm_list = Graph.CommunityPartition(tp_graph, colorScale="thermal")
n_comms   = len(set(comm_list))
print(f"Communities found: {n_comms}")
print(f"Community IDs    : {sorted(set(comm_list))}")

comm_by_nid = {}
for v, comm_val in zip(tp_verts_list, comm_list):
    key = (round(Vertex.X(v), 2), round(Vertex.Y(v), 2), round(Vertex.Z(v), 2))
    nid = vert_to_nid.get(key)
    if nid is not None:
        comm_by_nid[nid] = comm_val

# Community size distribution
comm_sizes = pd.Series(list(comm_by_nid.values())).value_counts().sort_index()
print("\nCommunity sizes:")
print(comm_sizes.to_string())

---
## 10. Shortest Paths

**Course API — S03-07:**  
`Graph.CompiledRoutingGraph(g, precomputeTurns=False)` — builds routing structure.  
`Graph.ShortestPath(crg, vertexA, vertexB)` — returns a `Wire` (path topology).

Three representative pairs are computed:
1. Two random room nodes (inter-apartment)
2. Room → nearest corridor on same floor
3. Corridor → core on same floor

In [ ]:
print("Building compiled routing graph (S03-07)...")
crg = Graph.CompiledRoutingGraph(tp_graph, precomputeTurns=False)
print("Routing graph ready.")

# Helper: look up vertex by node_id
nid_to_tpvert = {}
for v in tp_verts_list:
    d   = Topology.Dictionary(v)
    nid = Dictionary.ValueAtKey(d, "node_id")
    if nid is not None:
        nid_to_tpvert[int(nid)] = v

# Select example pairs
room_nids = nodes[nodes["node_role"] == "room"]["node_id"].astype(int).tolist()
corr_nids = nodes[nodes["node_role"] == "corridor"]["node_id"].astype(int).tolist()
core_nids = nodes[nodes["node_role"] == "core"]["node_id"].astype(int).tolist()

pairs = []
if len(room_nids) >= 2:
    pairs.append((room_nids[0], room_nids[-1], "room → room"))
if room_nids and corr_nids:
    pairs.append((room_nids[0], corr_nids[0], "room → corridor"))
if corr_nids and core_nids:
    pairs.append((corr_nids[0], core_nids[0], "corridor → core"))

path_results = []
for src_id, dst_id, label in pairs:
    vA = nid_to_tpvert.get(src_id)
    vB = nid_to_tpvert.get(dst_id)
    if vA is None or vB is None:
        print(f"  {label}: vertex not found in routing graph")
        continue
    path_wire = Graph.ShortestPath(crg, vA, vB)
    if path_wire is not None:
        n_edges = len(Topology.Edges(path_wire))
        print(f"  {label:20s}: path length = {n_edges} hops  (src={src_id}, dst={dst_id})")
        path_results.append({"label": label, "src": src_id, "dst": dst_id, "hops": n_edges})
    else:
        nx_path = nx.shortest_path(G_nx, src_id, dst_id) if nx.has_path(G_nx, src_id, dst_id) else None
        hops    = len(nx_path) - 1 if nx_path else None
        print(f"  {label:20s}: topologicpy path=None; nx path length={hops}")
        path_results.append({"label": label, "src": src_id, "dst": dst_id, "hops": hops})

---
## 11. Export Metrics Table

In [ ]:
metrics_rows = []
for _, row in nodes.iterrows():
    nid = int(row["node_id"])
    metrics_rows.append({
        "node_id"       : nid,
        "node_role"     : row.get("node_role", ""),
        "space_type"    : row.get("space_type", ""),
        "floor_id"      : row.get("floor_id"),
        "degree"        : G_nx.degree(nid) if nid in G_nx else 0,
        "degree_cent"   : round(dc_by_nid.get(nid, 0.0), 6),
        "betweenness"   : round(bc_by_nid.get(nid, 0.0), 6),
        "closeness"     : round(cc_by_nid.get(nid, 0.0), 6),
        "clustering"    : round(clustering.get(nid, 0.0), 6),
        "community"     : comm_by_nid.get(nid, -1),
    })

metrics_df   = pd.DataFrame(metrics_rows)
metrics_path = os.path.join(RESULTS_DIR, "metrics_table.csv")
metrics_df.to_csv(metrics_path, index=False)

print(f"metrics_table.csv : {len(metrics_df)} rows  →  {metrics_path}")
print()
print(metrics_df.describe())

---
## 12. Visualise Metrics

Each metric shown as a plan-view matplotlib scatter plot using XY centroids.
Brighter (YlOrRd) = higher metric value.

In [ ]:
pos_xy = {int(row["node_id"]): (float(row["X"]), float(row["Y"]))
          for _, row in nodes.iterrows()}

METRIC_SPECS = [
    ("degree_cent", "Degree Centrality",     "01_degree_centrality.png"),
    ("betweenness", "Betweenness Centrality", "02_betweenness_centrality.png"),
    ("closeness",   "Closeness Centrality",   "03_closeness_centrality.png"),
    ("clustering",  "Clustering Coefficient", "04_clustering_coefficient.png"),
]

for col, title, fname in METRIC_SPECS:
    vals    = metrics_df.set_index("node_id")[col].to_dict()
    v_list  = [vals.get(n, 0.0) for n in G_nx.nodes()]
    v_arr   = np.array(v_list, dtype=float)
    vmin, vmax = v_arr.min(), v_arr.max()
    norm    = Normalize(vmin=vmin, vmax=max(vmax, 1e-9))
    cmap    = plt.cm.YlOrRd
    colors  = [cmap(norm(val)) for val in v_list]

    fig, ax = plt.subplots(figsize=(12, 8))
    nx.draw_networkx_edges(G_nx, pos_xy, ax=ax, alpha=0.15, edge_color="#999", width=0.4)
    nx.draw_networkx_nodes(G_nx, pos_xy, ax=ax, node_color=colors, node_size=30,
                           edgecolors="none")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, shrink=0.35, label=title)
    ax.set_title(f"Double-L TB_01 — {title}", fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    out = os.path.join(VISUALS_DIR, fname)
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")

In [ ]:
# Community map
comm_cmap  = plt.cm.tab20
comm_vals  = metrics_df.set_index("node_id")["community"].to_dict()
comm_ids   = sorted(set(comm_vals.values()))
n_comm     = len(comm_ids)
comm_to_c  = {c: comm_cmap(i / max(n_comm, 1)) for i, c in enumerate(comm_ids)}
node_cols  = [comm_to_c.get(comm_vals.get(n, -1), "#CCC") for n in G_nx.nodes()]

fig, ax = plt.subplots(figsize=(12, 8))
nx.draw_networkx_edges(G_nx, pos_xy, ax=ax, alpha=0.15, edge_color="#999", width=0.4)
nx.draw_networkx_nodes(G_nx, pos_xy, ax=ax, node_color=node_cols, node_size=30,
                       edgecolors="none")
patches = [mpatches.Patch(color=comm_to_c[c], label=f"Comm {c}") for c in comm_ids[:16]]
ax.legend(handles=patches, loc="upper right", fontsize=7, ncol=2, framealpha=0.9)
ax.set_title("Double-L TB_01 — Community Detection (S03-08)", fontsize=12)
ax.axis("off")
plt.tight_layout()
out = os.path.join(VISUALS_DIR, "05_community_detection.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

---
## 13. Verification Summary

In [ ]:
expected_outputs = [
    os.path.join(RESULTS_DIR, "graph_summary.csv"),
    os.path.join(RESULTS_DIR, "metrics_table.csv"),
    os.path.join(VISUALS_DIR, "01_degree_centrality.png"),
    os.path.join(VISUALS_DIR, "02_betweenness_centrality.png"),
    os.path.join(VISUALS_DIR, "03_closeness_centrality.png"),
    os.path.join(VISUALS_DIR, "04_clustering_coefficient.png"),
    os.path.join(VISUALS_DIR, "05_community_detection.png"),
]

print("Output file check:")
all_ok = True
for path in expected_outputs:
    status = "OK" if os.path.exists(path) else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  {os.path.basename(path):45s}: {status}")

print()
if all_ok:
    print("All outputs present. Proceed to A2_02_DoubleL_Spatial_Interpretation.ipynb")
else:
    print("WARNING: Some outputs are missing. Scroll up and check for errors.")

---
## Final Submission Checklist

- [ ] `graph_summary.csv` saved with density, diameter, avg_path_len
- [ ] `metrics_table.csv` saved with 10 columns per node
- [ ] 5 PNG figures saved to `04_visuals/`
- [ ] Degree centrality: top nodes identified and their roles make spatial sense
- [ ] Betweenness centrality: corridor/core nodes rank higher than rooms
- [ ] Closeness centrality: centrally located rooms rank higher
- [ ] Clustering: rooms in tight rectangular arrangements score higher
- [ ] Communities: number of communities broadly matches floor count or wing count
- [ ] 3 shortest paths computed with hop counts logged

**Proceed to:** `A2_02_DoubleL_Spatial_Interpretation.ipynb`